# 레이저 그리드 품질검측 — 코랩

레이저 격자 이미지를 넣으면 **엑셀 조서 하나**가 나옵니다.

| 입력 | 필수? | 없으면 |
|---|---|---|
| 레이저 이미지 | **필수** | — |
| `camera_params.json` | 선택 | 사양 프로파일 값을 씁니다 |
| IMU | 선택 | **장비가 똑바로 서 있다고 가정**합니다 |
| 정답값 `cast_pixels.json` | 선택 | 검출 정확도·깊이 오차를 못 냅니다 |
| 장면 사진 (레이저 OFF) | 선택 | 레이저 이미지에서 선을 지워 배경으로 씁니다 |


## 1. 설치

In [ ]:
import os
if os.path.isdir('laser_grid/.git'):
    !git -C laser_grid pull -q          # 두 번째부터는 최신본만 당긴다
else:
    !git clone -q https://github.com/znlsl10-rgb/laser_grid.git
%cd /content/laser_grid
!pip install -q -r requirements.txt
!apt-get -qq install -y fonts-nanum > /dev/null 2>&1   # 한글 폰트
!git log --oneline -1
print('준비 완료')


## 2. 파일 올리기

왼쪽 파일 탭에 끌어다 놓아도 되고, 아래 셀로 올려도 됩니다.

In [ ]:
from google.colab import files
up = files.upload()          # 이미지 (+ 있으면 json 들)
print(list(up))

## 3. 실행

`image` 만 필수입니다. 나머지는 있으면 넣고, 없으면 지우세요.

In [ ]:
import os
from run_pipeline import run

def find(name):
    """파일 탭에 끌어다 놓으면 /content 에, 업로드 셀로 올리면 여기에 온다.
    둘 다 찾아본다. 없으면 None — 선택 입력이면 그대로 넘어간다."""
    if not name:
        return None
    for c in (name, f'/content/{name}', f'/content/laser_grid/{name}'):
        if os.path.exists(c):
            return c
    print(f'  [없음] {name} — 이 입력 없이 진행합니다')
    return None

res = run(
    image  = find('CAST.png'),              # 필수 — 레이저 격자 이미지
    params = find('camera_params.json'),    # 선택 — 카메라 사양
    truth  = find('cast_pixels.json'),      # 선택 — 정답값
    imu    = None,                          # 선택 — 없으면 똑바로 섰다고 가정
    scene_image = find('CAM.png'),          # 선택 — 레이저 OFF 사진
    out    = '/content/결과/',              # 폴더를 줘도 되고 .xlsx 를 줘도 된다
)

print('조서:', res['xlsx'])


### IMU 를 줄 때

셋 중 아무 형식이나 됩니다.

```python
imu = {'pitch_deg': 34.0, 'roll_deg': 0.0}   # 아래로 숙인 각 / 광축 둘레 회전
imu = {'gravity': [0, 0.83, 0.56]}          # 조사기 좌표계 중력 벡터
imu = {'accel':   [0, -0.83, -0.56]}        # 정지 상태 가속도계 읽음
```

**IMU 를 안 주면 장비가 똑바로 서 있다고 가정합니다.** 수직도는 면 법선과 중력의 사잇각이라, 장비가 실제로 1° 기울어 있었다면 판정도 1° 틀립니다 — 허용치가 ±0.5° 인 것을 생각하면 작지 않습니다.

## 4. 결과 보기

In [ ]:
import pandas as pd
pd.read_excel(res['xlsx'], sheet_name='6.검측결과')

### 부재별 3D 요약 — 폭이 어디서 왔는지 함께 본다

`폭 근거` 칸을 꼭 보세요. 부재 폭은 상수가 아니라 **가로선 끊김에서 잰 값**입니다.

| 폭 근거 | 뜻 |
|---|---|
| `단면 반지름` | 격자선이 2줄 이상 걸려 단면이 풀렸다 — 가장 믿을 만함 |
| `가로선 끊김 폭` | 부재가 배경에 드리운 그림자 폭에서 지름을 되돌렸다 |
| `같은 장면의 다른 부재 폭` | 이 부재는 못 쟀고 같은 장면의 다른 부재 값을 빌렸다 |
| `기본값(폭 미측정)` | 아무것도 못 쟀다 — 분할 경계를 그대로 믿지 말 것 |


In [ ]:
pd.read_excel(res['xlsx'], sheet_name='9.3D좌표(부재별)')[
    ['부재', '클래스', '검측', '격자선 수', '측정 점수', '가로선 점수',
     '부재 반폭(mm)', '폭 근거']]


In [ ]:
from IPython.display import Image, display
for k in ('3D점군', '세그멘테이션', '선검출대조'):
    p = res['images'].get(k)
    if p:
        print(k); display(Image(p, width=900))

## 5. 내려받기

In [ ]:
files.download(res['xlsx'])

---
## 엑셀 조서 구성

| 시트 | 내용 |
|---|---|
| 1.요약 | 입력·가정·결과 한눈에 |
| 2.설계값 | 사양과 그 출처 등급 |
| **3.선검출(1단계)** | 선별 검출 결과 + 정답 대조 + 대조 그림 |
| **4.깊이검증(2단계)** | 화소→3D 깊이, 정답이 있으면 mm 오차표 |
| **5.세그멘테이션(3단계)** | 색↔부재 대응 + 그림 |
| **6.검측결과** | 부재별 수직도·수평도·평활도 판정 |
| 7~8 | 평활도 근거 · 요철 위치 + 그림 |
| **9.3D좌표(부재별)** | 3D 산점도(부재별 색) + 점수·중심·크기·부재 폭 |
| 10.3D좌표(점목록) | 부재 구분이 붙은 좌표 목록 |
| 11.유의사항 | 이 값을 어디까지 믿을 수 있는가 |

## 판정에 '판정보류' 가 나오면

부재에 세로 격자선이 **한 줄만** 걸린 경우입니다. 한 줄에서 나온 3D 점은
그 레이저 평면 안에 놓이므로, 위·아래 화소의 깊이 차이로 잡히는 것은
**카메라 쪽으로 넘어진 성분**뿐입니다. 옆으로 넘어진 성분은 깊이를 전혀
바꾸지 않아 보이지 않습니다.

그래서 잰 값은 참값의 **하한**이고, 판정이 한쪽으로만 성립합니다.

| 잰 값 | 판정 | 왜 |
|---|---|---|
| 허용치 초과 | **기준초과 (확정)** | 참값은 하한보다 크니 뒤집힐 수 없다 |
| 허용치 이내 | **판정보류(단면 미분해)** | 못 본 성분이 넘었을 수 있다 |

부재 뒤에 배경이 있으면 가로선 끊김(그림자)에서 옆 성분을 되찾아 정상
판정으로 돌아옵니다. 조서 비고에 `가로선 화소 끊김` 또는 `가림 그림자
실루엣` 으로 적힙니다. 근본 해결은 부재에 세로선이 2줄 이상 걸리도록
더 가까이서 찍거나 격자를 조밀하게 하는 것입니다.

## 정확도에 대해

선검출의 한계는 알고리즘이 아니라 **입력 이미지**가 정합니다. 선이 안티에일리어싱 없이 이진(0/255)으로 그려져 있으면 선 중심이 0.5px 격자에 갇혀, 어떤 추정기를 써도 σ = 1/√12 = **0.289px** 아래로 못 내려갑니다.

파이프라인이 이 하한을 자동으로 재서 조서에 적습니다.
